# Laboratorium: matching keypointów i panorama stitching

## Cel zajęć
Na poprzednich zajęciach implementowaliście **detektor Harrisa**. Dzisiaj wykorzystamy go (i inne detektory):
1. zbudujemy prosty pipeline **Harris + deskryptor oparty o wycinki obrazu + matching**,
2. porównamy go z **SIFT-em** z OpenCV,
3. sprawdzimy metodę nowoczesną z biblioteki: **Tiny RoMa / RoMa**,
4. użyjemy dopasowań do estymacji **homografii** i złożenia **prostej panoramy**.

Dla każdej metody policzymy:
- liczbę punktów / korespondencji,
- liczbę **inlierów po algorytmie dopasowującym RANSAC**,
- czas działania,
- jakość panoramy / zgodność homografii.

## Powtórka z wykładu
- **Harris** jest detektorem, nie deskryptorem — dlatego do niego dołożymy prosty deskryptor oparty o znormalizowany wycinek obrazu.
- **SIFT** daje zarówno punkty, jak i deskryptory.

Ponadto wykorzystamy bibliotekę z nowoczesnym podejściem opartym o sieć neuronową: **RoMa**

## Dane
Notebook działa w dwóch trybach:
- jeśli w katalogu `data/` są pliki `img1.jpg` i `img2.jpg`, użyje ich,
- w przeciwnym razie wczyta przykładowe dane z bilioteki scikit, wygeneruje syntetyczną parę: dokona transformacji obrazu tak, aby para obrazów tylko częściowo przedstawiała ten sam fragment sceny.



In [ ]:
import time
import math
from pathlib import Path
from tempfile import TemporaryDirectory

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sympy import maximum

plt.rcParams["figure.figsize"] = (10, 6)

SEED = 7
np.random.seed(SEED)


In [ ]:
!pip install romatch

In [ ]:
from skimage import data as skdata

def imread_rgb(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def to_gray(img_rgb):
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

def generate_synthetic_pair():
    base = skdata.astronaut()
    base = cv2.resize(base, (960, 640), interpolation=cv2.INTER_AREA)

    h, w = base.shape[:2]
    src = np.float32([
        [0, 0],
        [w - 1, 0],
        [w - 1, h - 1],
        [0, h - 1],
    ])
    dst = np.float32([
        [40, 20],
        [w - 80, 10],
        [w - 20, h - 30],
        [70, h - 10],
    ])
    H = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(base, H, (w, h))

    img1 = base[:, :700].copy()
    img2 = warped[:, 220:920].copy()
    return img1, img2, "synthetic"

def load_pair():
    p1 = Path("data/img1.jpg")
    p2 = Path("data/img2.jpg")
    if p1.exists() and p2.exists():
        img1 = imread_rgb(p1)
        img2 = imread_rgb(p2)
        source = "data/img1.jpg + data/img2.jpg"
    else:
        img1, img2, source = generate_synthetic_pair()
    return img1, img2, source

img1, img2, data_source = load_pair()
gray1, gray2 = to_gray(img1), to_gray(img2)

print("Źródło danych:", data_source)
print("img1:", img1.shape, "img2:", img2.shape)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(img1)
ax[0].set_title("Obraz 1")
ax[0].axis("off")
ax[1].imshow(img2)
ax[1].set_title("Obraz 2")
ax[1].axis("off")
plt.show()


In [ ]:

def show_keypoints(img_rgb, keypoints, title="", max_show=300):
    pts = np.array(keypoints, dtype=np.float32)
    if len(pts) > max_show:
        idx = np.random.choice(len(pts), size=max_show, replace=False)
        pts = pts[idx]
    plt.figure(figsize=(8, 6))
    plt.imshow(img_rgb)
    if len(pts):
        plt.scatter(pts[:, 0], pts[:, 1], s=10)
    plt.title(title)
    plt.axis("off")
    plt.show()

def draw_matches_canvas(img1_rgb, img2_rgb, pts1, pts2, inlier_mask=None, max_show=80, title=""):
    pts1 = np.asarray(pts1, dtype=np.float32)
    pts2 = np.asarray(pts2, dtype=np.float32)

    if len(pts1) == 0:
        print("Brak matchów do wizualizacji.")
        return

    n = len(pts1)
    if n > max_show:
        idx = np.random.choice(n, max_show, replace=False)
        pts1, pts2 = pts1[idx], pts2[idx]
        if inlier_mask is not None:
            inlier_mask = np.asarray(inlier_mask).astype(bool)[idx]

    h1, w1 = img1_rgb.shape[:2]
    h2, w2 = img2_rgb.shape[:2]
    H = max(h1, h2)
    W = w1 + w2

    canvas = np.zeros((H, W, 3), dtype=np.uint8)
    canvas[:h1, :w1] = img1_rgb
    canvas[:h2, w1:w1+w2] = img2_rgb

    plt.figure(figsize=(14, 7))
    plt.imshow(canvas)
    for i, (p1, p2) in enumerate(zip(pts1, pts2)):
        x1, y1 = p1
        x2, y2 = p2
        x2s = x2 + w1
        if inlier_mask is None:
            style = dict(linewidth=1.0, alpha=0.8)
            point_alpha = 0.8
        else:
            ok = bool(inlier_mask[i])
            style = dict(linewidth=1.2 if ok else 0.7, alpha=0.9 if ok else 0.35)
            point_alpha = 0.9 if ok else 0.35
        plt.plot([x1, x2s], [y1, y2], **style)
        plt.scatter([x1, x2s], [y1, y2], s=10, alpha=point_alpha)
    plt.title(title)
    plt.axis("off")
    plt.show()


In [ ]:
def estimate_homography_and_metrics(pts1, pts2, ransac_thresh=3.0):
    pts1 = np.asarray(pts1, dtype=np.float32)
    pts2 = np.asarray(pts2, dtype=np.float32)

    if len(pts1) < 4:
        return {
            "H": None,
            "mask": None,
            "n_matches": int(len(pts1)),
            "n_inliers": 0,
            "inlier_ratio": 0.0,
        }

    H, mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, ransac_thresh)
    if mask is None:
        n_inliers = 0
        inlier_ratio = 0.0
    else:
        mask = mask.ravel().astype(bool)
        n_inliers = int(mask.sum())
        inlier_ratio = float(mask.mean())

    return {
        "H": H,
        "mask": mask,
        "n_matches": int(len(pts1)),
        "n_inliers": n_inliers,
        "inlier_ratio": inlier_ratio,
    }

def stitch_panorama(img1_rgb, img2_rgb, H_12):
    if H_12 is None:
        raise ValueError("Brak homografii.")

    h1, w1 = img1_rgb.shape[:2]
    h2, w2 = img2_rgb.shape[:2]

    corners1 = np.float32([[0, 0], [w1, 0], [w1, h1], [0, h1]]).reshape(-1, 1, 2)
    corners1_w = cv2.perspectiveTransform(corners1, H_12)

    corners2 = np.float32([[0, 0], [w2, 0], [w2, h2], [0, h2]]).reshape(-1, 1, 2)
    all_corners = np.vstack([corners1_w, corners2])

    xmin, ymin = np.floor(all_corners.min(axis=0).ravel()).astype(int)
    xmax, ymax = np.ceil(all_corners.max(axis=0).ravel()).astype(int)

    tx = -xmin if xmin < 0 else 0
    ty = -ymin if ymin < 0 else 0

    T = np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]], dtype=np.float64)
    out_w = int(xmax - xmin)
    out_h = int(ymax - ymin)

    pano1 = cv2.warpPerspective(img1_rgb, T @ H_12, (out_w, out_h))
    pano2 = np.zeros_like(pano1)
    pano2[ty:ty+h2, tx:tx+w2] = img2_rgb

    mask1 = (pano1.sum(axis=2) > 0).astype(np.float32)[..., None]
    mask2 = (pano2.sum(axis=2) > 0).astype(np.float32)[..., None]

    denom = np.clip(mask1 + mask2, 1e-6, None)
    blended = ((pano1.astype(np.float32) * mask1 + pano2.astype(np.float32) * mask2) / denom).astype(np.uint8)
    return blended


## 1. Harris + prosty deskryptor oparty o łatki

In [3]:
# TODO: uzupełnij pipeline Harris + łatki.
# 1) oblicz odpowiedź Harrisa,
# 2) zrób non-maximum suppression,
# 3) wytnij znormalizowane łatki jako deskryptory,
# 4) dopasuj deskryptory ratio testem,
# 5) policz homografię i inliery.

import numpy as np
import cv2

def harris_response(gray, block_size=2, ksize=3, k=0.04):
    #obliczenie gradientu przy użyciu Sobela
    ix = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=ksize)
    iy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=ksize)

    ix2 = ix**2
    iy2 = iy**2

    ixiy = ix * iy

    #sumowanie w oknie przy użyciu rozmycia pudelkowego
    W = (block_size, block_size)
    s_ix2 = cv2.blur(ix2, W)
    s_iy2 = cv2.blur(iy2, W)
    s_ixiy = cv2.blur(ixiy, W)

    #wyznacznik i slad macierzy
    det = s_ix2 * s_iy2 - s_ixiy**2
    trace = s_ix2 + s_iy2

    return det - k * trace**2

def select_keypoints(response, max_points=800, threshold_rel=0.01, min_dist=8):
    # 1. Ustal próg względem maksimum
    thresh = threshold_rel * np.max(response)

    # 2. Wyzeruj wszystko poniżej progu
    r_thresh = np.copy(response)
    r_thresh[r_thresh < thresh] = 0

    # 3. Wykonaj NMS (Non-Maximum Suppression) przy użyciu dylatacji
    # Szukamy pikseli, które są równe maksymalnej wartości w swoim sąsiedztwie
    local_max = cv2.dilate(r_thresh, np.ones((min_dist, min_dist)))
    peaks = (r_thresh == local_max) & (r_thresh > 0)

    y, x = np.where(peaks)
    scores = r_thresh[y, x]

    # 4. Posortuj kandydatów malejąco po sile odpowiedzi
    indices = np.argsort(-scores)
    candidates = list(zip(x[indices], y[indices]))

    # 5. Zachłannie wybierz punkty odległe od siebie o min_dist
    selected = []
    for x_c, y_c in candidates:
        keep = True
        for x_s, y_s in selected:
            if np.sqrt((x_c - x_s)**2 + (y_c - y_s)**2) < min_dist:
                keep = False
                break
        if keep:
            selected.append((x_c, y_c))

        if len(selected) >= max_points:
            break

    return np.array(selected)

def match_descriptors(desc1, desc2, ratio=0.75, mutual=True):
    if len(desc1) == 0 or len(desc2) < 2:
        return np.empty((0, 2), dtype=np.int32)

    a2 = np.sum(desc1 ** 2, axis=1, keepdims=True)
    b2 = np.sum(desc2 ** 2, axis=1, keepdims=True).T

    # TODO 1: Macierz kwadratów odległości
    d2 = a2 + b2 - 2 * np.dot(desc1, desc2.T)
    d2 = np.maximum(d2, 0) # numeryczna stabilność

    # TODO 2: Indeksy 2 najbliższych sąsiadów
    nn12 = np.argsort(d2, axis=1)[:, :2]

    # TODO 3: Odległości
    best = d2[np.arange(len(d2)), nn12[:, 0]]
    second = d2[np.arange(len(d2)), nn12[:, 1]]

    # TODO 4: Ratio test Lowe'a
    keep = best < (ratio**2) * second

    matches = np.stack([np.arange(len(desc1)), nn12[:, 0]], axis=1)
    matches = matches[keep]

    if mutual and len(matches):
        # TODO 5: Sprawdzenie wzajemności (dla desc2 najlepszym musi być desc1)
        nn21 = np.argmin(d2, axis=0)
        mutual_mask = np.array([nn21[j] == i for i, j in matches], dtype=bool)
        matches = matches[mutual_mask]

    return matches.astype(np.int32)

def patch_descriptors(gray, keypoints, patch_size=11):
    assert patch_size % 2 == 1, "patch_size musi być nieparzyste"
    r = patch_size // 2

    valid_kp = []
    descs = []

    gray = gray.astype(np.float32)

    for x, y in np.asarray(keypoints, dtype=np.int32):
        if x - r < 0 or y - r < 0 or x + r >= gray.shape[1] or y + r >= gray.shape[0]:
            continue
        patch = gray[y-r:y+r+1, x-r:x+r+1].copy()
        patch -= patch.mean()
        patch /= (patch.std() + 1e-6)
        valid_kp.append([x, y])
        descs.append(patch.reshape(-1))

    if len(descs) == 0:
        return np.empty((0, 2), dtype=np.float32), np.empty((0, patch_size*patch_size), dtype=np.float32)

    return np.array(valid_kp, dtype=np.float32), np.array(descs, dtype=np.float32)

def match_descriptors(desc1, desc2, ratio=0.75, mutual=True):
    if len(desc1) == 0 or len(desc2) < 2:
        return np.empty((0, 2), dtype=np.int32)

    a2 = np.sum(desc1 ** 2, axis=1, keepdims=True)
    b2 = np.sum(desc2 ** 2, axis=1, keepdims=True).T

    # TODO 1:
    # policz macierz kwadratów odległości:
    # d2 = a2 + b2 - 2 * desc1 @ desc2.T
    d2 = ...

    # TODO 2:
    # weź indeksy 2 najbliższych sąsiadów w każdym wierszu d2
    nn12 = ...

    # TODO 3:
    # odczytaj odległość do najlepszego i drugiego najlepszego dopasowania
    best = ...
    second = ...

    # TODO 4:
    # ratio test Lowe'a: best < (ratio**2) * second
    keep = ...

    matches = np.stack([np.arange(len(desc1)), nn12[:, 0]], axis=1)
    matches = matches[keep]

    if mutual and len(matches):
        # TODO 5:
        # dla każdego deskryptora z desc2 znajdź najlepszy indeks w desc1
        nn21 = ...
        mutual_mask = np.array([nn21[j] == i for i, j in matches], dtype=bool)
        matches = matches[mutual_mask]

    return matches.astype(np.int32)

# Przykładowe wywołanie — odkomentuj po implementacji.
# resp1 = harris_response(gray1)
# resp2 = harris_response(gray2)
# kp1 = select_keypoints(resp1)
# kp2 = select_keypoints(resp2)
# kp1, desc1 = patch_descriptors(gray1, kp1, patch_size=11)
# kp2, desc2 = patch_descriptors(gray2, kp2, patch_size=11)
# matches = match_descriptors(desc1, desc2, ratio=0.80, mutual=True)
# pts1 = kp1[matches[:, 0]]
# pts2 = kp2[matches[:, 1]]
# harris_result = estimate_homography_and_metrics(pts1, pts2)
# draw_matches_canvas(img1, img2, pts1, pts2, harris_result["mask"], title="Harris + patch")


error: OpenCV(4.13.0) D:/a/opencv-python/opencv-python/opencv/modules/imgproc/src/filter.simd.hpp:3098: error: (-213:The function/feature is not implemented) Unsupported combination of buffer format (=6), and destination format (=5) in function 'cv::opt_AVX2::getLinearColumnFilter'


## 2. SIFT z OpenCV

In [ ]:
def sift_match(gray1, gray2, ratio=0.75, nfeatures=2000):
    sift = cv2.SIFT_create(nfeatures=nfeatures)
    kp1, des1 = sift.detectAndCompute(gray1, None)
    kp2, des2 = sift.detectAndCompute(gray2, None)

    pts1 = np.array([k.pt for k in kp1], dtype=np.float32)
    pts2 = np.array([k.pt for k in kp2], dtype=np.float32)

    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    knn = bf.knnMatch(des1, des2, k=2)

    good = []
    for pair in knn:
        if len(pair) < 2:
            continue
        m, n = pair
        if m.distance < ratio * n.distance:
            good.append((m.queryIdx, m.trainIdx, m.distance))

    if len(good):
        idx1 = np.array([g[0] for g in good], dtype=np.int32)
        idx2 = np.array([g[1] for g in good], dtype=np.int32)
        mpts1 = pts1[idx1]
        mpts2 = pts2[idx2]
    else:
        mpts1 = np.empty((0, 2), dtype=np.float32)
        mpts2 = np.empty((0, 2), dtype=np.float32)

    return pts1, pts2, mpts1, mpts2

t0 = time.perf_counter()
sift_kp1, sift_kp2, pts1_s, pts2_s = sift_match(gray1, gray2, ratio=0.75, nfeatures=2500)
sift_result = estimate_homography_and_metrics(pts1_s, pts2_s)
sift_time = time.perf_counter() - t0

print("SIFT keypoints:", len(sift_kp1), len(sift_kp2))
print("SIFT matches:", sift_result["n_matches"])
print("SIFT inliers:", sift_result["n_inliers"])
print("SIFT inlier ratio:", round(sift_result["inlier_ratio"], 3))
print("SIFT runtime [s]:", round(sift_time, 3))

show_keypoints(img1, sift_kp1, title="SIFT keypoints - obraz 1")
show_keypoints(img2, sift_kp2, title="SIFT keypoints - obraz 2")
draw_matches_canvas(img1, img2, pts1_s, pts2_s, sift_result["mask"], title="SIFT + ratio test")


## 3. Tiny RoMa / RoMa

In [ ]:
def roma_match(img1_rgb, img2_rgb, sample_count=1000):
    try:
        import torch
        import inspect
        from romatch import tiny_roma_v1_outdoor
    except Exception as e:
        print("Sekcja RoMa pominięta. Zainstaluj pakiet `romatch`.")
        print("Błąd importu:", repr(e))
        return None

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = tiny_roma_v1_outdoor(device=device)

    with TemporaryDirectory() as tmp:
        p1 = Path(tmp) / "img1.png"
        p2 = Path(tmp) / "img2.png"
        cv2.imwrite(str(p1), cv2.cvtColor(img1_rgb, cv2.COLOR_RGB2BGR))
        cv2.imwrite(str(p2), cv2.cvtColor(img2_rgb, cv2.COLOR_RGB2BGR))

        t0 = time.perf_counter()

        match_kwargs = {}
        try:
            if "device" in inspect.signature(model.match).parameters:
                match_kwargs["device"] = device
        except Exception:
            pass

        warp, certainty = model.match(str(p1), str(p2), **match_kwargs)
        matches, cert = model.sample(warp, certainty, num=sample_count)
        h1, w1 = img1_rgb.shape[:2]
        h2, w2 = img2_rgb.shape[:2]
        kpts1, kpts2 = model.to_pixel_coordinates(matches, h1, w1, h2, w2)
        runtime = time.perf_counter() - t0

    pts1 = kpts1.detach().cpu().numpy().astype(np.float32)
    pts2 = kpts2.detach().cpu().numpy().astype(np.float32)
    result = estimate_homography_and_metrics(pts1, pts2)

    return {
        "pts1": pts1,
        "pts2": pts2,
        "result": result,
        "runtime": runtime,
        "device": device,
    }

roma_out = roma_match(img1, img2, sample_count=1200)

if roma_out is not None:
    print("Tiny RoMa device:", roma_out["device"])
    print("Tiny RoMa matches:", roma_out["result"]["n_matches"])
    print("Tiny RoMa inliers:", roma_out["result"]["n_inliers"])
    print("Tiny RoMa inlier ratio:", round(roma_out["result"]["inlier_ratio"], 3))
    print("Tiny RoMa runtime [s]:", round(roma_out["runtime"], 3))

    draw_matches_canvas(
        img1, img2,
        roma_out["pts1"], roma_out["pts2"],
        roma_out["result"]["mask"],
        max_show=120,
        title="Tiny RoMa / RoMa"
    )


## 4. Porównanie metod

In [ ]:
rows = [
    {
        "metoda": "Harris + patch",
        "n_matches": harris_result["n_matches"],
        "n_inliers": harris_result["n_inliers"],
        "inlier_ratio": harris_result["inlier_ratio"],
        "runtime_s": harris_time,
    },
    {
        "metoda": "SIFT",
        "n_matches": sift_result["n_matches"],
        "n_inliers": sift_result["n_inliers"],
        "inlier_ratio": sift_result["inlier_ratio"],
        "runtime_s": sift_time,
    },
]

if "roma_out" in globals() and roma_out is not None:
    rows.append(
        {
            "metoda": "Tiny RoMa",
            "n_matches": roma_out["result"]["n_matches"],
            "n_inliers": roma_out["result"]["n_inliers"],
            "inlier_ratio": roma_out["result"]["inlier_ratio"],
            "runtime_s": roma_out["runtime"],
        }
    )

df = pd.DataFrame(rows)
df = df.sort_values(["n_inliers", "inlier_ratio"], ascending=False).reset_index(drop=True)
df


## 5. Panorama

In [ ]:
candidates = [
    ("Harris + patch", harris_result),
    ("SIFT", sift_result),
]

if "roma_out" in globals() and roma_out is not None:
    candidates.append(("Tiny RoMa", roma_out["result"]))

best_name, best_result = max(
    candidates,
    key=lambda x: (x[1]["n_inliers"], x[1]["inlier_ratio"])
)

print("Wybrana metoda do panoramy:", best_name)

if best_result["H"] is not None:
    pano = stitch_panorama(img1, img2, best_result["H"])
    plt.figure(figsize=(14, 7))
    plt.imshow(pano)
    plt.title(f"Panorama ({best_name})")
    plt.axis("off")
    plt.show()
else:
    print("Nie udało się oszacować homografii.")


## Źródła

- OpenCV: Harris Corner Detection  
  https://docs.opencv.org/4.x/dc/d0d/tutorial_py_features_harris.html

- OpenCV: SIFT  
  https://docs.opencv.org/4.x/da/df5/tutorial_py_sift_intro.html

- OpenCV: Feature Matching  
  https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html

- OpenCV: Feature Matching + Homography  
  https://docs.opencv.org/4.x/d1/de0/tutorial_py_feature_homography.html

- OpenCV: Stitcher / homography model for panoramas  
  https://docs.opencv.org/4.x/d8/d19/tutorial_stitcher.html

- RoMa (official repository)  
  https://github.com/Parskatt/RoMa
